In [32]:
dict_source = {"source": 0, "key 2": 2}

keys = dict_source.keys()
print(keys)

dict_keys(['source', 'key 2'])


In [33]:
key = next(iter(dict_source))
print(key)

source


In [34]:
import torch

# Tensor ngẫu nhiên (random)
tensor = torch.randn(3)  # shape (3,4)

print(tensor)

tensor([-0.5475, -0.1174,  0.3972])


In [35]:
tensor.shape[0]

3

In [36]:
from typing import Any, Dict, List, OrderedDict
import torch
import numpy as np

class Instance(OrderedDict):
    def __init__(self, **kwargs):
        super().__init__(kwargs)

    def __setattr__(self, key, value):
        self[key] = value

    def __getattr__(self, key):
        try:
            return self[key]
        except KeyError:
            raise AttributeError(f"{key} not found")

    def get_fields(self):
        """Get current attributes/fields registered under the sample.

        Returns:
            List[str]: Attributes registered under the Sample.

        """
        return list(self.keys())

class InstanceList(OrderedDict):
    def __init__(self, instance_list: List["Instance"] = [], pad_value=0):
        super().__init__(self)
        
        self.pad_value = pad_value

        if len(instance_list) == 0:
            return

        assert all(isinstance(i, Instance) for i in instance_list)
      
        for key in instance_list[0].get_fields():
            
            values = [instance.get(key) for instance in instance_list]
           
            v0 = values[0]
                
            
            if isinstance(v0, np.ndarray):
                values = [torch.tensor(value) for value in values]
                values = self.pad_values(values)
                values = torch.cat(values, dim=0)
            if isinstance(v0, torch.Tensor):
            
                values = self.pad_values(values)
                
                values = torch.cat(values, dim=0)
            elif hasattr(type(v0), "cat"):
                values = type(v0).cat(values)
            else:
                values
            self.set(key, values)

    def __setattr__(self, name: str, val: Any) -> None:
        if name.startswith("_"):
            super().__setattr__(name, val)
        else:
            self.set(name, val)

    def __getattr__(self, name: str) -> Any:
        if name == "_fields" or name not in self:
            raise AttributeError(f"{name} not found")
        return self[name]

    def set(self, name: str, value: Any) -> None:
        """
        Set the field named `name` to `value`.
        The length of `value` must be the number of Instance,
        and must agree with other existing fields in this object.
        """
        self[name] = value

    def has(self, name: str) -> bool:
        """
        Returns:
            bool: whether the field called `name` exists.
        """
        return name in self

    def remove(self, name: str) -> None:
        """
        Remove the field called `name`.
        """
        del self[name]

    def get(self, name: str) -> Any:
        """
        Returns the field called `name`.
        """
        return self[name]

    def get_fields(self) -> Dict[str, Any]:
        """
        Returns:
            dict: a dict which maps names (str) to data of the fields

        Modifying the returned dict will modify this instance.
        """
        return list(self.keys())

    @property
    def batch_size(self) -> int:
        for k in self.keys():
            if isinstance(self[k], torch.Tensor):
                return self[k].shape[0]
            if isinstance(self[k], list):
                return len(self[k])

        return 0

    # Tensor-like methods
    def to(self, *args: Any, **kwargs: Any) -> "InstanceList":
        """
        Returns:
            InstanceList: all fields are called with a `to(device)`, if the field has this method.
        """
        ret = InstanceList()
        for k, v in self.items():
            if hasattr(v, "to"):
                v = v.to(*args, **kwargs)
            ret.set(k, v)
        return ret

    # Tensor-like methods
    def unsqueeze(self, *args: Any, **kwargs) -> "InstanceList":
        """
        Returns:
            InstanceList: all fields are called with a `unsqueeze(dim)`, if the field has this method.
        """
        ret = InstanceList()
        for k, v in self.items():
            if hasattr(v, "unsqueeze"):
                v = v.unsqueeze(*args, **kwargs)
            ret.set(k, v)
        
        return ret

    # Tensor-like methods
    def squeeze(self, *args: Any, **kwargs) -> "InstanceList":
        """
        Returns:
            InstanceList: all fields are called with a `squeeze(dim)`, if the field has this method.
        """
        ret = InstanceList()
        for k, v in self.items():
            if hasattr(v, "squeeze"):
                v = v.squeeze(*args, **kwargs)
            ret.set(k, v)
        
        return ret

    # special method for concatenating tensor objects
    def pad_values(self, values: List[torch.tensor]) -> List[torch.tensor]:
        
        padded_values = []
        max_len = max([value.shape[0] for value in values])

        for value in values:
            
          
            additional_len = max_len - value.shape[0]
            
            if additional_len == 0:
                padded_values.append(value.unsqueeze(0))
                continue
            # add another dimension to pad the right dimension
            if len(value.shape) == 2: 
                
                padding_tensor = torch.zeros((additional_len, value.shape[1])).long().fill_(self.pad_value)
            else: 
                padding_tensor = torch.zeros((additional_len, )).long().fill_(self.pad_value)
      
            value = torch.cat([value, padding_tensor], dim=0)
        
            padded_values.append(value.unsqueeze(0))
            
        
        return padded_values

    def __str__(self) -> str:
        s = self.__class__.__name__ + "("
        s += "fields=[{}])".format(", ".join((f"{k}: {v}" for k, v in self.items())))
        return s

    __repr__ = __str__

In [37]:
if __name__ == "__main__":
    print("=== TẠO DỮ LIỆU MẪU (Mô phỏng Dataset) ===")
    
    # Mẫu 1: Câu dài trung bình (5 tokens)
    inst1 = Instance(
        id="doc_01",
        input_ids=torch.tensor([1, 15, 20, 89, 2]),       # <bos> A B C <eos>
        input_type_ids=torch.tensor([1, 1, 1, 0, 0]),     # Source=1, Target=0
        src_len=torch.tensor([3])                         # Chiều dài Source: 3
    )

    # Mẫu 2: Câu rất ngắn (3 tokens)
    inst2 = Instance(
        id="doc_02",
        input_ids=torch.tensor([1, 45, 2]),               # <bos> D <eos>
        input_type_ids=torch.tensor([1, 1, 0]),
        src_len=torch.tensor([2])
    )

    # Mẫu 3: Câu dài nhất (7 tokens)
    inst3 = Instance(
        id="doc_03",
        input_ids=torch.tensor([1, 10, 11, 12, 13, 14, 2]), 
        input_type_ids=torch.tensor([1, 1, 1, 1, 0, 0, 0]),
        src_len=torch.tensor([4])
    )

    print("Đã tạo 3 Instance riêng lẻ với độ dài: 5, 3, 7.")
    
    # --- GOM BATCH BẰNG INSTANCELIST ---
    print("\n=== ĐƯA VÀO INSTANCELIST ĐỂ GOM BATCH ===")
    # Giả sử pad_idx của vocab bạn là 0
    batch = InstanceList([inst1, inst2, inst3], pad_value=0)

    # --- IN KẾT QUẢ ĐỂ KIỂM TRA ---
    print(f"Batch Size tự nhận diện: {batch.batch_size}")
    
    print("\n1. Field 'id' (Kiểu chuỗi thuần - sẽ được giữ nguyên dạng List):")
    print(batch.id)

    print("\n2. Field 'input_ids' (Được đệm số 0 cho bằng câu dài nhất là 7):")
    print(batch.input_ids)

    print("\n3. Field 'input_type_ids' (Cũng được đệm tương ứng):")
    print(batch.input_type_ids)

    print("\n4. Field 'src_len' (Các tensor [1] được gộp thành cột [Batch, 1]):")
    print(batch.src_len)
    
    print("\n5. Thử gọi hàm .to('cuda') (Sẽ không lỗi nếu máy bạn có GPU, hoặc thử đổi thành .to('cpu')):")
    batch = batch.to('cpu')
    print("Chuyển device thành công!")

=== TẠO DỮ LIỆU MẪU (Mô phỏng Dataset) ===
Đã tạo 3 Instance riêng lẻ với độ dài: 5, 3, 7.

=== ĐƯA VÀO INSTANCELIST ĐỂ GOM BATCH ===
Batch Size tự nhận diện: 3

1. Field 'id' (Kiểu chuỗi thuần - sẽ được giữ nguyên dạng List):
['doc_01', 'doc_02', 'doc_03']

2. Field 'input_ids' (Được đệm số 0 cho bằng câu dài nhất là 7):
tensor([[ 1, 15, 20, 89,  2,  0,  0],
        [ 1, 45,  2,  0,  0,  0,  0],
        [ 1, 10, 11, 12, 13, 14,  2]])

3. Field 'input_type_ids' (Cũng được đệm tương ứng):
tensor([[1, 1, 1, 0, 0, 0, 0],
        [1, 1, 0, 0, 0, 0, 0],
        [1, 1, 1, 1, 0, 0, 0]])

4. Field 'src_len' (Các tensor [1] được gộp thành cột [Batch, 1]):
tensor([[3],
        [2],
        [4]])

5. Thử gọi hàm .to('cuda') (Sẽ không lỗi nếu máy bạn có GPU, hoặc thử đổi thành .to('cpu')):
Chuyển device thành công!


In [38]:
print("Shape của input_ids:", batch.input_ids.shape)
# Kết quả: torch.Size([3, 7]) -> (Batch_size=3, Max_length=7)

print("Shape của input_type_ids:", batch.input_type_ids.shape)
# Kết quả: torch.Size([3, 7])

print("Shape của src_len:", batch.src_len.shape)
# Kết quả: torch.Size([3, 1])

# Riêng field 'id' là list string, không có .shape, ta dùng len()
print("Độ dài của list ID:", len(batch.id))
# Kết quả: 3

Shape của input_ids: torch.Size([3, 7])
Shape của input_type_ids: torch.Size([3, 7])
Shape của src_len: torch.Size([3, 1])
Độ dài của list ID: 3


In [39]:
src_len = batch.src_len

In [ ]:
# Chuyển toàn bộ Tensor thành List Python
danh_sach_chieu_dai = batch.src_len.tolist()

[[3], [2], [4]]
[3, 2, 4]
